In [0]:
import time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra

nodes = (spark.table("workspace.spb.silver_walk_nodes")
         .select("node_id", "node_idx", "lat", "lon").toPandas())
edges = (spark.table("workspace.spb.silver_walk_edges")
         .select("src_idx", "dst_idx", "length_m").toPandas())

N = len(nodes)
assert nodes.node_idx.max() == N - 1

graph = csr_matrix(
    (edges.length_m.to_numpy(),
     (edges.src_idx.to_numpy(), edges.dst_idx.to_numpy())),
    shape=(N, N),
)
print(N, "nodes,", graph.nnz, "stored edges")

In [0]:
bar_nodes = (spark.table("workspace.spb.snapped_walk_nodes").select("node_id").distinct()
             .join(spark.table("workspace.spb.silver_walk_nodes")
                        .select("node_id", "node_idx"), "node_id")
             .toPandas().sort_values("node_idx").reset_index(drop=True))
targets = bar_nodes.node_idx.to_numpy()
print(len(targets))

In [0]:
sources = targets[:5]

t0 = time.time()
D = dijkstra(graph, directed=False, indices=sources)    # shape (5, N)
secs = time.time() - t0

print(f"{secs:.1f} s for 5 sources | array {D.shape} = {D.nbytes/1e6:.0f} MB")
print(f"extrapolated to {len(targets)} sources: {secs/5*len(targets)/60:.1f} min")

D_bars = D[:, targets]       # keep only distances to the bar nodes: shape (5, 1203)

In [0]:
# 1. no unreachable pairs (graph is one component)
print("inf values:", np.isinf(D_bars).sum())

# 2. distance to self is zero, and the matrix is symmetric
print("self distances:", [D_bars[i, i] for i in range(5)])
print("symmetry:", D_bars[0, 1], D_bars[1, 0])

# 3. walking distance vs straight-line distance
def hav_m(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * 6371000 * np.arcsin(np.sqrt(a))

xy = nodes.set_index("node_idx").loc[targets, ["lat", "lon"]].to_numpy()
straight = hav_m(xy[0, 0], xy[0, 1], xy[:, 0], xy[:, 1])        # from source 0 to every bar
mask = straight > 50                                             # skip itself and same-block bars
ratio = D_bars[0][mask] / straight[mask]
print("walk/straight ratio  min, median, 99th pct:", np.percentile(ratio, [0, 50, 99]))

In [0]:
n = len(targets)
M = np.empty((n, n))                 # 1203 x 1203, about 12 MB
B = 100                              # sources per batch, about 300 MB per batch

t0 = time.time()
for i in range(0, n, B):
    batch = dijkstra(graph, directed=False, indices=targets[i:i + B])
    M[i:i + B] = batch[:, targets]   # drop the other 378k columns right away
print(f"{time.time() - t0:.0f} s")

assert not np.isinf(M).any()
assert np.allclose(M, M.T)           # symmetric
assert (np.diag(M) == 0).all()

In [0]:
ids = bar_nodes.node_id.to_numpy()   # same order as targets
ii, jj = np.indices((n, n))
long_df = pd.DataFrame({
    "src_node_id": ids[ii.ravel()],
    "dst_node_id": ids[jj.ravel()],
    "dist_m": M.ravel(),
})
(spark.createDataFrame(long_df)
    .write.format("delta").mode("overwrite")
    .saveAsTable("workspace.spb.gold_node_distances"))